In [1]:
import sys
import pickle

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
TRIALS = 2
FIX_FILE_PATH = "./import_fix.py"
for _ in range(TRIALS):
    try:
      from src import PersonalAI, PersonalAIConfig, QAPipelineConfig, MemPipelineConfig, \
            GraphModelConfig, EmbeddingsModelConfig, EmbedderModelConfig

      from src.db_drivers import KeyValueDriverConfig, GraphDriverConfig, VectorDriverConfig
      from src.db_drivers.kv_driver import DEFAULT_INMEMORYKV_CONFIG
      from src.db_drivers.graph_driver import DEFAULT_INMEMORYGRAPH_CONFIG
      from src.db_drivers.vector_driver import VectorDBConnectionConfig

      from src.qa_pipeline.knowledge_retriever import AStarGraphSearchConfig, AStarMetricsConfig, BFSSearchConfig, MixturedGraphSearchConfig
      from src.qa_pipeline import QueryLLMParserConfig, KnowledgeComparatorConfig, KnowledgeRetrieverConfig, QALLMGeneratorConfig

      from src.memorize_pipeline import LLMExtractorConfig, LLMUpdatorConfig

      from src.utils import NodeType, Logger
    except RuntimeError as e:
        from pathlib import Path
        fix_path = Path(FIX_FILE_PATH)
        if fix_path.is_file():
            %run {fix_path} --base_dir BASEDIR
        else:
            raise e

c:\Users\nikit\anaconda3\envs\LLM\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Загружем датасет с триплетами, на основе которого будет построен граф знаний

In [3]:
PKL_GRAPH_PATH = '../../data/pickled_graphs/DiaasqGigachat.pickle'

with open(PKL_GRAPH_PATH, 'rb') as f:
    formated_triplets = pickle.load(f)

In [4]:
length = 211542

print(len(formated_triplets))
formated_triplets = formated_triplets[:length]
print(len(formated_triplets))

211542
211542


#### 2. Задаём конфигурацию графа знаний

In [5]:
# Graph model configuration
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
GRAPH_MODEL_CONFIG = GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG)

In [ ]:
# Vector model configuration
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing' # TO CHANGE
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing' # TO CHANGE
NEED_TO_CLEAR = True

VECTOR_NODES_STORAGE_CONFIG = VectorDriverConfig(db_config=VectorDBConnectionConfig(path=NODES_DB_PATH, need_to_clear=NEED_TO_CLEAR))
VECTOR_TRIPLETS_STIRAGE_CONFIG = VectorDriverConfig(db_config=VectorDBConnectionConfig(path=TRIPLETS_DB_PATH, need_to_clear=NEED_TO_CLEAR))

DEVICE = 'cuda' # TO CHANGE
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small' # TO CHANGE
EMBEDDER_MODEL_CONFIG = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)

VECTOR_MODEL_CONFIG = EmbeddingsModelConfig(
    nodesdb_driver_config=VECTOR_NODES_STORAGE_CONFIG,
    tripletsdb_driver_config=VECTOR_TRIPLETS_STIRAGE_CONFIG,
    embedder_config=EMBEDDER_MODEL_CONFIG)

In [ ]:
# QA-pipeline retrieve stage configuration (configuring mixture graph search/retriever)
KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)
ASTAR_RETRIEVER_CONFIG = AStarGraphSearchConfig(
    metrics_config=AStarMetricsConfig(
        h_metric_name='ip', # TO CHANGE 
        kvdriver_config=KV_STORAGE_CONFIG),
    max_depth=20, max_passed_nodes=1000, # TO CHANGE
    accepted_node_types=[NodeType.object , NodeType.hyper, NodeType.episodic]) # TO CHANGE

BFS_RETRIEVER_CONFIG = BFSSearchConfig(
    strict_filter = True, hyper_episodic_num = 15, # TO CHANGE
    chain_triplets_num = 25, other_triplets_num = 6) # TO CHANGE

RETRIEVER_NAME = 'mixture'
RETRIEVER_CONFIG = MixturedGraphSearchConfig(
    astar_config=ASTAR_RETRIEVER_CONFIG,
    bfs_config=BFS_RETRIEVER_CONFIG
)

In [ ]:
LANGUAGE = 'en' # TO CHANGE ('ru' | 'en' | 'auto')

In [ ]:
# QA-pipeline configuration
QA_PIPELINE_CONFIG = QAPipelineConfig(
    query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
    knowledge_comparator_config=KnowledgeComparatorConfig(),
    knowledge_retriever_config=KnowledgeRetrieverConfig(
        retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_CONFIG),
    answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE))

# Memorize-pipeline configuration
MEM_PIPELINE_CONFIG = MemPipelineConfig(
    xtractor_config=LLMExtractorConfig(lang=LANGUAGE),
    updator_config=LLMUpdatorConfig(lang=LANGUAGE))

PERSONALAI_CONFIG = PersonalAIConfig(
    graph_struct_config=GRAPH_MODEL_CONFIG,
    embedds_struct_config=VECTOR_MODEL_CONFIG,
    qa_pipeline_config=QA_PIPELINE_CONFIG,
    mem_pipeline_config=MEM_PIPELINE_CONFIG,
    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [ ]:
personalai = PersonalAI(config=PERSONALAI_CONFIG)

#### 4. Добавляем в граф загруженные триплеты

In [8]:
print("uploading data to graph-storage")
personalai.kg_model.graph_struct.create_triplets(formated_triplets)

100%|██████████| 398/398 [02:31<00:00,  2.62it/s]


In [ ]:
print("uploading data to vector-storage")
personalai.kg_model.embeddings_struct.create_triplets(formated_triplets)

#### 5. Q&A

In [9]:
qa_examples = [
  ("Matthew has positive, negative or neutral opinion about game of Mi 10pro on 22.12.2018?",
  "Positive"),
  ("Lily has positive, negative or neutral opinion about power of XiaoMi on 4.7.2019?",
  "Negative"),
  ("What opinion (positive, negative or neutral) about scheduling of IQOO9 was last during Zachary's experience of IQOO9?",
  "positive"),
  ("What opinion (positive, negative or neutral) about electricity of Apple was last during Jessica's experience of Apple?",
  "positive"),
  ("What Abraham's opinion (positive, negative or neutral) about fast charging of Xiaomi was dominant during using Xiaomi?",
  "positive"),
  ("Do Adrian and Herbert have any common devices (which Adrian and Herbert both use)? If so, list common devices. Otherwise, answer 'No'.",
  "No"),
  ("Do Bernard and Bailey have any common devices (which Bernard and Bailey both use)? If so, list common devices. Otherwise, answer 'No'.",
  "Apple"),
  ("Which people have negative opinion about battery of Apple phone on 15.9.2018?",
  "Hugh")
  ]

In [10]:
for question in qa_examples:
    answer, info = personalai.answer_question(question[0])
    print("MODEL ANSWER: ", answer)
    print("TRUE ANSWER: ", question[1])
    print("=" * 35)

MODEL ANSWER:  На основании предоставленной информации нельзя сделать вывод о мнении Мэттью относительно игры на Mi 10pro 22 декабря 2018 года, так как в тексте нет прямых высказываний Мэттью по этому поводу.
TRUE ANSWER:  Positive
MODEL ANSWER:  На основе предоставленной информации, мнение Лили о мощности Xiaomi на 4 июля 2019 года является нейтральным. Это можно предположить из комментария Джорджа от 4 июля 2019 года, в котором он говорит, что нет ничего плохого в Xiaomi, за исключением потребления энергии. Однако прямого высказывания Лили о мощности Xiaomi не было найдено.
TRUE ANSWER:  Negative
MODEL ANSWER:  Негативное мнение о расписании IQOO9 было последним во время опыта Захари с IQOO9.
TRUE ANSWER:  positive
MODEL ANSWER:  Последнее мнение о электричестве Apple было нейтральным во время опыта Джессики с Apple.
TRUE ANSWER:  positive
MODEL ANSWER:  Доступная информация не содержит прямого мнения Абрахама о быстрой зарядке Xiaomi, которое было бы доминирующим во время использова